In [ ]:
!pip install -q transformers datasets scikit-learn

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "CUDA version:",
        torch.version.cuda
    )
else:
    print("ERROR: CUDA GPU not available.")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0


In [7]:
# ============================================================
# CUDA BENCHMARK
# TEDBench CATH CLASSIFICATION
# ESM-2 (320) + STRUCTURAL FEATURES (28) = 348
# ============================================================

!pip -q install datasets transformers accelerate scikit-learn

import os
import time
import json
import random
import warnings

import numpy as np
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

TRAIN_SIZE = 5000
VAL_SIZE = 1500
TEST_SIZE = 1500

MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

BATCH_SIZE = 2
MAX_LENGTH = 1022

EPOCHS = 40
LEARNING_RATE = 0.001

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("CUDA BENCHMARK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# LOAD TEDBENCH
# ============================================================

print("\n" + "=" * 70)
print("LOADING TEDBENCH")
print("=" * 70)

dataset = load_dataset("TEDBench/ted")

train_ds = dataset["train"].shuffle(seed=SEED).select(
    range(TRAIN_SIZE)
)

val_ds = dataset["val"].shuffle(seed=SEED).select(
    range(VAL_SIZE)
)

test_ds = dataset["test"].shuffle(seed=SEED).select(
    range(TEST_SIZE)
)

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))

print("Columns:", train_ds.column_names)

# ============================================================
# CATH CLASS
# ============================================================

def decode_cath_class(split, label):

    cath_label = split.features["label"].int2str(
        int(label)
    )

    return str(cath_label).split(".")[0]


def get_cath_labels(split):

    return np.array([
        decode_cath_class(split, x["label"])
        for x in split
    ])


y_train_text = get_cath_labels(train_ds)
y_val_text = get_cath_labels(val_ds)
y_test_text = get_cath_labels(test_ds)

classes = np.array(
    sorted(np.unique(y_train_text))
)

class_to_id = {
    c: i
    for i, c in enumerate(classes)
}

y_train = np.array([
    class_to_id[x]
    for x in y_train_text
])

print("\nCATH Class labels:")
print(classes)

print("Class mapping:", class_to_id)

NUM_CLASSES = len(classes)

print("Number of CATH classes:", NUM_CLASSES)

# ============================================================
# ESM-2
# ============================================================

print("\n" + "=" * 70)
print("LOADING ESM-2")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

esm_model = AutoModel.from_pretrained(
    MODEL_NAME
)

esm_model = esm_model.to(device)
esm_model.eval()

print("ESM-2 loaded.")

# ============================================================
# ESM EMBEDDING EXTRACTION
# ============================================================

def generate_esm_embeddings(
    split,
    name
):

    print("\n" + "-" * 70)
    print("ESM EMBEDDINGS:", name)
    print("-" * 70)

    embeddings = []

    start = time.perf_counter()

    with torch.no_grad():

        for start_idx in range(
            0,
            len(split),
            BATCH_SIZE
        ):

            batch = split[
                start_idx:
                start_idx + BATCH_SIZE
            ]

            sequences = batch["sequence"]

            tokens = tokenizer(
                sequences,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt"
            )

            tokens = {
                k: v.to(device)
                for k, v in tokens.items()
            }

            outputs = esm_model(
                **tokens
            )

            hidden = outputs.last_hidden_state

            attention_mask = tokens[
                "attention_mask"
            ]

            mask = attention_mask.unsqueeze(
                -1
            ).float()

            masked_hidden = (
                hidden * mask
            )

            pooled = (
                masked_hidden.sum(dim=1)
                /
                mask.sum(dim=1).clamp(
                    min=1e-9
                )
            )

            embeddings.append(
                pooled.cpu().numpy()
            )

    embeddings = np.concatenate(
        embeddings,
        axis=0
    ).astype(np.float32)

    elapsed = (
        time.perf_counter()
        - start
    )

    print(
        f"{name} ESM time: "
        f"{elapsed:.3f} seconds"
    )

    print(
        "Embedding shape:",
        embeddings.shape
    )

    return embeddings, elapsed


esm_train, esm_train_time = generate_esm_embeddings(
    train_ds,
    "train"
)

esm_val, esm_val_time = generate_esm_embeddings(
    val_ds,
    "validation"
)

esm_test, esm_test_time = generate_esm_embeddings(
    test_ds,
    "test"
)

esm_total_time = (
    esm_train_time
    + esm_val_time
    + esm_test_time
)

print(
    "\nTotal ESM extraction:",
    round(esm_total_time, 3),
    "seconds"
)

# ============================================================
# STRUCTURAL FEATURES
# ============================================================
#
# 28 FEATURES:
#
# 1  contact_count
# 2  contact_density
# 3  contact_order
# 4  radius_of_gyration
# 5  end_to_end_distance
# 6  ca_distance_mean
# 7  ca_distance_std
# 8  ca_distance_min
# 9  ca_distance_max
# 10 ca_distance_q25
# 11 ca_distance_median
# 12 ca_distance_q75
# 13 n_ca_distance_mean
# 14 ca_c_distance_mean
# 15 c_n_distance_mean
# 16 pairwise_distance_mean
# 17 pairwise_distance_std
# 18 pairwise_distance_max
# 19 pairwise_distance_q25
# 20 pairwise_distance_median
# 21 pairwise_distance_q75
# 22 ca_x_std
# 23 ca_y_std
# 24 ca_z_std
# 25 compactness
# 26 ca_radius_mean
# 27 ca_radius_std
# 28 ca_radius_max
#
# ============================================================

def extract_structural_features(coords):

    coords = np.asarray(
        coords,
        dtype=np.float32
    )

    if coords.ndim != 3:
        return np.zeros(
            28,
            dtype=np.float32
        )

    if coords.shape[1:] != (3, 3):
        return np.zeros(
            28,
            dtype=np.float32
        )

    length = coords.shape[0]

    if length < 2:
        return np.zeros(
            28,
            dtype=np.float32
        )

    # --------------------------------------------------------
    # BACKBONE ATOMS
    # --------------------------------------------------------

    N = coords[:, 0, :]
    CA = coords[:, 1, :]
    C = coords[:, 2, :]

    # --------------------------------------------------------
    # CA BOND DISTANCES
    # --------------------------------------------------------

    ca_bonds = np.linalg.norm(
        CA[1:] - CA[:-1],
        axis=1
    )

    ca_mean = np.mean(ca_bonds)
    ca_std = np.std(ca_bonds)
    ca_min = np.min(ca_bonds)
    ca_max = np.max(ca_bonds)

    ca_q25 = np.percentile(
        ca_bonds,
        25
    )

    ca_median = np.median(
        ca_bonds
    )

    ca_q75 = np.percentile(
        ca_bonds,
        75
    )

    # --------------------------------------------------------
    # BACKBONE BOND GEOMETRY
    # --------------------------------------------------------

    n_ca = np.linalg.norm(
        N - CA,
        axis=1
    )

    ca_c = np.linalg.norm(
        CA - C,
        axis=1
    )

    c_n = np.linalg.norm(
        C[:-1] - N[1:],
        axis=1
    )

    n_ca_mean = np.mean(n_ca)
    ca_c_mean = np.mean(ca_c)
    c_n_mean = np.mean(c_n)

    # --------------------------------------------------------
    # GEOMETRIC CENTER
    # --------------------------------------------------------

    center = np.mean(
        CA,
        axis=0
    )

    radial_distances = np.linalg.norm(
        CA - center,
        axis=1
    )

    radius_of_gyration = np.sqrt(
        np.mean(
            radial_distances ** 2
        )
    )

    end_to_end_distance = np.linalg.norm(
        CA[-1] - CA[0]
    )

    # --------------------------------------------------------
    # CONTACT FEATURES
    # --------------------------------------------------------

    # Use pairwise CA distances
    if length > 1:

        contacts = 0
        contact_sum = 0.0

        for i in range(length):

            if i + 2 >= length:
                continue

            distances = np.linalg.norm(
                CA[i + 2:] - CA[i],
                axis=1
            )

            contact_mask = distances < 8.0

            contacts += np.sum(
                contact_mask
            )

            if np.any(contact_mask):

                contact_sum += np.sum(
                    np.abs(
                        np.arange(
                            i + 2,
                            length
                        )[contact_mask]
                        - i
                    )
                )

        contact_count = float(
            contacts
        )

        total_possible = (
            length * (length - 1) / 2
        )

        contact_density = (
            contact_count
            /
            max(
                total_possible,
                1.0
            )
        )

        if contact_count > 0:

            contact_order = (
                contact_sum
                /
                contact_count
                /
                max(length, 1)
            )

        else:

            contact_order = 0.0

    else:

        contact_count = 0.0
        contact_density = 0.0
        contact_order = 0.0

    # --------------------------------------------------------
    # PAIRWISE DISTANCES
    # --------------------------------------------------------

    pairwise_values = []

    # Sample pairwise distances for
    # memory efficiency

    max_samples = 5000

    if length > 1:

        total_pairs = (
            length * (length - 1) // 2
        )

        if total_pairs <= max_samples:

            for i in range(length - 1):

                d = np.linalg.norm(
                    CA[i + 1:] - CA[i],
                    axis=1
                )

                pairwise_values.extend(
                    d.tolist()
                )

        else:

            rng = np.random.default_rng(
                42
            )

            for _ in range(
                max_samples
            ):

                i = rng.integers(
                    0,
                    length - 1
                )

                j = rng.integers(
                    i + 1,
                    length
                )

                d = np.linalg.norm(
                    CA[j] - CA[i]
                )

                pairwise_values.append(
                    float(d)
                )

    pairwise_values = np.asarray(
        pairwise_values,
        dtype=np.float32
    )

    if len(pairwise_values) > 0:

        pairwise_mean = np.mean(
            pairwise_values
        )

        pairwise_std = np.std(
            pairwise_values
        )

        pairwise_max = np.max(
            pairwise_values
        )

        pairwise_q25 = np.percentile(
            pairwise_values,
            25
        )

        pairwise_median = np.median(
            pairwise_values
        )

        pairwise_q75 = np.percentile(
            pairwise_values,
            75
        )

    else:

        pairwise_mean = 0.0
        pairwise_std = 0.0
        pairwise_max = 0.0
        pairwise_q25 = 0.0
        pairwise_median = 0.0
        pairwise_q75 = 0.0

    # --------------------------------------------------------
    # COORDINATE SPREAD
    # --------------------------------------------------------

    ca_x_std = np.std(
        CA[:, 0]
    )

    ca_y_std = np.std(
        CA[:, 1]
    )

    ca_z_std = np.std(
        CA[:, 2]
    )

    # --------------------------------------------------------
    # COMPACTNESS
    # --------------------------------------------------------

    compactness = (
        radius_of_gyration
        /
        max(
            end_to_end_distance,
            1e-6
        )
    )

    # --------------------------------------------------------
    # EXTRA RADIAL FEATURES
    # --------------------------------------------------------

    ca_radius_mean = np.mean(
        radial_distances
    )

    ca_radius_std = np.std(
        radial_distances
    )

    ca_radius_max = np.max(
        radial_distances
    )

    # --------------------------------------------------------
    # FINAL 28 FEATURES
    # --------------------------------------------------------

    features = np.array(
        [

            contact_count,
            contact_density,
            contact_order,

            radius_of_gyration,
            end_to_end_distance,

            ca_mean,
            ca_std,
            ca_min,
            ca_max,
            ca_q25,
            ca_median,
            ca_q75,

            n_ca_mean,
            ca_c_mean,
            c_n_mean,

            pairwise_mean,
            pairwise_std,
            pairwise_max,
            pairwise_q25,
            pairwise_median,
            pairwise_q75,

            ca_x_std,
            ca_y_std,
            ca_z_std,

            compactness,

            ca_radius_mean,
            ca_radius_std,
            ca_radius_max

        ],
        dtype=np.float32
    )

    return features


# ============================================================
# EXTRACT STRUCTURAL FEATURES FOR A SPLIT
# ============================================================

def generate_structural_features(
    split,
    name
):

    print("\n" + "-" * 70)
    print("STRUCTURAL FEATURES:", name)
    print("-" * 70)

    start = time.perf_counter()

    features = []

    for i, row in enumerate(split):

        feature = extract_structural_features(
            row["coords"]
        )

        features.append(feature)

    features = np.asarray(
        features,
        dtype=np.float32
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    print(
        f"{name} structural time: "
        f"{elapsed:.3f} seconds"
    )

    print(
        "Structural shape:",
        features.shape
    )

    return features, elapsed


struct_train, struct_train_time = (
    generate_structural_features(
        train_ds,
        "train"
    )
)

struct_val, struct_val_time = (
    generate_structural_features(
        val_ds,
        "validation"
    )
)

struct_test, struct_test_time = (
    generate_structural_features(
        test_ds,
        "test"
    )
)

struct_total_time = (
    struct_train_time
    + struct_val_time
    + struct_test_time
)

print(
    "\nTotal structural extraction:",
    round(struct_total_time, 3),
    "seconds"
)

# ============================================================
# VERIFY EXACT DIMENSIONS
# ============================================================

print("\n" + "=" * 70)
print("FEATURE DIMENSIONS")
print("=" * 70)

print(
    "ESM:",
    esm_train.shape[1]
)

print(
    "Structural:",
    struct_train.shape[1]
)

assert esm_train.shape[1] == 320
assert struct_train.shape[1] == 28

# ============================================================
# COMBINE FEATURES
# ============================================================

X_train = np.concatenate(
    [
        esm_train,
        struct_train
    ],
    axis=1
)

X_val = np.concatenate(
    [
        esm_val,
        struct_val
    ],
    axis=1
)

X_test = np.concatenate(
    [
        esm_test,
        struct_test
    ],
    axis=1
)

print(
    "Combined:",
    X_train.shape
)

assert X_train.shape[1] == 348

# ============================================================
# STANDARDIZATION
# ============================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
).astype(np.float32)

X_val = scaler.transform(
    X_val
).astype(np.float32)

X_test = scaler.transform(
    X_test
).astype(np.float32)

# ============================================================
# PYTORCH DATA
# ============================================================

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

# ============================================================
# MLP CLASSIFIER
# ============================================================

class ProteinClassifier(
    torch.nn.Module
):

    def __init__(
        self,
        input_dim,
        num_classes
    ):

        super().__init__()

        self.network = torch.nn.Sequential(

            torch.nn.Linear(
                input_dim,
                512
            ),

            torch.nn.ReLU(),

            torch.nn.BatchNorm1d(
                512
            ),

            torch.nn.Dropout(
                0.30
            ),

            torch.nn.Linear(
                512,
                256
            ),

            torch.nn.ReLU(),

            torch.nn.BatchNorm1d(
                256
            ),

            torch.nn.Dropout(
                0.30
            ),

            torch.nn.Linear(
                256,
                num_classes
            )
        )

    def forward(self, x):

        return self.network(x)


model = ProteinClassifier(
    input_dim=348,
    num_classes=NUM_CLASSES
).to(device)

# ============================================================
# CLASS WEIGHTS
# ============================================================

class_counts = np.bincount(
    y_train,
    minlength=NUM_CLASSES
)

class_weights = (
    len(y_train)
    /
    (
        NUM_CLASSES
        *
        np.maximum(
            class_counts,
            1
        )
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

# ============================================================
# LOSS + OPTIMIZER
# ============================================================

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

# ============================================================
# MOVE TRAINING DATA TO GPU
# ============================================================

X_train_gpu = X_train_tensor.to(
    device
)

y_train_gpu = y_train_tensor.to(
    device
)

X_val_gpu = X_val_tensor.to(
    device
)

# ============================================================
# TRAINING
# ============================================================

print("\n" + "=" * 70)
print("GPU CLASSIFIER TRAINING")
print("=" * 70)

classifier_start = time.perf_counter()

for epoch in range(
    1,
    EPOCHS + 1
):

    model.train()

    optimizer.zero_grad()

    outputs = model(
        X_train_gpu
    )

    loss = criterion(
        outputs,
        y_train_gpu
    )

    loss.backward()

    optimizer.step()

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    with torch.no_grad():

        val_outputs = model(
            X_val_gpu
        )

        val_predictions = (
            torch.argmax(
                val_outputs,
                dim=1
            )
            .cpu()
            .numpy()
        )

    val_acc = accuracy_score(
        y_val_text,
        classes[val_predictions]
    )

    val_balanced = (
        balanced_accuracy_score(
            y_val_text,
            classes[val_predictions]
        )
    )

    if (
        epoch == 1
        or epoch % 5 == 0
    ):

        print(
            f"Epoch {epoch:02d} | "
            f"Loss {loss.item():.4f} | "
            f"Val Acc {val_acc:.4f} | "
            f"Val Balanced {val_balanced:.4f}"
        )

# Synchronize GPU before timing ends

if torch.cuda.is_available():

    torch.cuda.synchronize()

classifier_time = (
    time.perf_counter()
    - classifier_start
)

# ============================================================
# TEST
# ============================================================

model.eval()

with torch.no_grad():

    test_outputs = model(
        X_test_tensor.to(device)
    )

    test_predictions = (
        torch.argmax(
            test_outputs,
            dim=1
        )
        .cpu()
        .numpy()
    )

test_pred_labels = classes[
    test_predictions
]

# ============================================================
# METRICS
# ============================================================

test_accuracy = accuracy_score(
    y_test_text,
    test_pred_labels
)

test_balanced_accuracy = (
    balanced_accuracy_score(
        y_test_text,
        test_pred_labels
    )
)

test_macro_f1 = f1_score(
    y_test_text,
    test_pred_labels,
    average="macro"
)

test_weighted_f1 = f1_score(
    y_test_text,
    test_pred_labels,
    average="weighted"
)

# ============================================================
# TOTAL RUNTIME
# ============================================================

if torch.cuda.is_available():

    torch.cuda.synchronize()

total_runtime = (
    esm_total_time
    + struct_total_time
    + classifier_time
)

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL CUDA RESULTS")
print("=" * 70)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print(
    "ESM features:",
    esm_train.shape[1]
)

print(
    "Structural features:",
    struct_train.shape[1]
)

print(
    "Combined features:",
    X_train.shape[1]
)

print(
    "ESM extraction:",
    round(
        esm_total_time,
        3
    ),
    "seconds"
)

print(
    "Structural extraction:",
    round(
        struct_total_time,
        3
    ),
    "seconds"
)

print(
    "Classifier training:",
    round(
        classifier_time,
        3
    ),
    "seconds"
)

print(
    "Total runtime:",
    round(
        total_runtime,
        3
    ),
    "seconds"
)

print(
    "Validation accuracy:",
    round(
        val_acc,
        4
    )
)

print(
    "Test accuracy:",
    round(
        test_accuracy,
        4
    )
)

print(
    "Balanced accuracy:",
    round(
        test_balanced_accuracy,
        4
    )
)

print(
    "Macro F1:",
    round(
        test_macro_f1,
        4
    )
)

print(
    "Weighted F1:",
    round(
        test_weighted_f1,
        4
    )
)

# ============================================================
# SAVE RESULTS
# ============================================================

os.makedirs(
    "cuda_benchmark_results",
    exist_ok=True
)

results = {

    "device": str(device),

    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "CPU"
    ),

    "train_size": TRAIN_SIZE,
    "validation_size": VAL_SIZE,
    "test_size": TEST_SIZE,

    "esm_features": int(
        esm_train.shape[1]
    ),

    "structural_features": int(
        struct_train.shape[1]
    ),

    "combined_features": int(
        X_train.shape[1]
    ),

    "esm_time_seconds": float(
        esm_total_time
    ),

    "structural_time_seconds": float(
        struct_total_time
    ),

    "classifier_training_seconds": float(
        classifier_time
    ),

    "total_runtime_seconds": float(
        total_runtime
    ),

    "validation_accuracy": float(
        val_acc
    ),

    "test_accuracy": float(
        test_accuracy
    ),

    "test_balanced_accuracy": float(
        test_balanced_accuracy
    ),

    "test_macro_f1": float(
        test_macro_f1
    ),

    "test_weighted_f1": float(
        test_weighted_f1
    )
}

with open(
    "cuda_benchmark_results/"
    "cuda_cath_class_results.json",
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )

np.save(
    "cuda_benchmark_results/"
    "cuda_train_features.npy",
    X_train
)

np.save(
    "cuda_benchmark_results/"
    "cuda_val_features.npy",
    X_val
)

np.save(
    "cuda_benchmark_results/"
    "cuda_test_features.npy",
    X_test
)

print("\nSaved:")
print(
    "cuda_benchmark_results/"
    "cuda_cath_class_results.json"
)

print(
    "cuda_benchmark_results/"
    "cuda_train_features.npy"
)

print(
    "cuda_benchmark_results/"
    "cuda_val_features.npy"
)

print(
    "cuda_benchmark_results/"
    "cuda_test_features.npy"
)

print("\nDONE.")

CUDA BENCHMARK
PyTorch: 2.11.0+cu130
CUDA available: True
Device: cuda
GPU: Tesla T4

LOADING TEDBENCH
Train: 5000
Validation: 1500
Test: 1500
Columns: ['name', 'sequence', 'coords', 'plddt', 'residue_index', 'seq_ids', 'label']

CATH Class labels:
['1' '2' '3' '4']
Class mapping: {np.str_('1'): 0, np.str_('2'): 1, np.str_('3'): 2, np.str_('4'): 3}
Number of CATH classes: 4

LOADING ESM-2


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ESM-2 loaded.

----------------------------------------------------------------------
ESM EMBEDDINGS: train
----------------------------------------------------------------------
train ESM time: 49.037 seconds
Embedding shape: (5000, 320)

----------------------------------------------------------------------
ESM EMBEDDINGS: validation
----------------------------------------------------------------------
validation ESM time: 15.091 seconds
Embedding shape: (1500, 320)

----------------------------------------------------------------------
ESM EMBEDDINGS: test
----------------------------------------------------------------------
test ESM time: 14.638 seconds
Embedding shape: (1500, 320)

Total ESM extraction: 78.766 seconds

----------------------------------------------------------------------
STRUCTURAL FEATURES: train
----------------------------------------------------------------------
train structural time: 275.713 seconds
Structural shape: (5000, 28)

--------------------------